In [1]:
import pandas as pd

data=pd.read_csv(r"C:\Users\HEMANATH\Desktop\customer-churn-prediction\data\processed\Cleaned_data_for_eda.csv")
data.drop("Unnamed: 0",axis=1,inplace=True)
data

,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,...,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value
0,33.964131,-118.272783,Male,No,No,No,2,Yes,No,DSL,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
1,34.059281,-118.307420,Female,No,No,Yes,2,Yes,No,Fiber optic,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
2,34.048013,-118.293953,Female,No,No,Yes,8,Yes,Yes,Fiber optic,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1
3,34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,...,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1
4,34.039224,-118.266293,Male,No,No,Yes,49,Yes,Yes,Fiber optic,...,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7027,34.341737,-116.539416,Female,No,No,No,72,Yes,No,No,...,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Bank transfer (automatic),21.15,1419.40,0
7028,34.667815,-117.536183,Male,No,Yes,Yes,24,Yes,Yes,DSL,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,0
7029,34.559882,-115.637164,Female,No,Yes,Yes,72,Yes,Yes,Fiber optic,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,0
7030,34.167800,-116.864330,Female,No,Yes,Yes,11,No,No phone service,DSL,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,0


In [2]:
df_fe=data.copy()

In [3]:
df_fe["TenureGroup"]=pd.cut(df_fe["Tenure Months"],bins=[0,12,24,48,72],labels=["New", "Mid", "Long", "Very Long"])

In [4]:
df_fe["SpendCategory"]=pd.cut(df_fe["Monthly Charges"],bins=[0,35,70,100,150],labels=["Low","Medium","High","VeryHigh"])
df_fe["SpendCategory"].unique()

['Medium', 'High', 'VeryHigh', 'Low']
Categories (4, object): ['Low' < 'Medium' < 'High' < 'VeryHigh']

In [5]:
df_fe["AvgSpendPerMonth"]=(df_fe["Total Charges"]/(df_fe["Monthly Charges"]+1))
df_fe["AvgSpendPerMonth"]

0        1.971741
1        2.115063
2        8.152012
3       28.790643
4       48.102197
          ...    
7027    64.081264
7028    23.199301
7029    70.661228
7030    11.321895
7031    64.177215
Name: AvgSpendPerMonth, Length: 7032, dtype: float64

In [6]:
df_fe["HighRiskCustomer"]=(
    (df_fe["Contract"]=="Month-to-Month") &
    (df_fe["Tenure Months"]<12)
)

In [7]:
service_cols=[
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support"
]

df_fe["ServiceCount"]=(
    df_fe[service_cols]=="Yes"
).sum(axis=1)

In [8]:
stream_cols=[
    "Streaming TV", "Streaming Movies"
]

df_fe["StreamingCount"]=(
    df_fe[stream_cols]=="Yes").sum(axis=1)

In [9]:
df_fe["IsElectronicCheck"]=(df_fe["Payment Method"]=="Electronic check").astype(int)

In [10]:
df_fe["IsAutoPay"]=df_fe["Payment Method"].isin([
    "Bank transfer (automatic)",
    "Credit card (automatic)"
]).astype(int)

In [11]:
df_fe["TotalEngagement"]=(
    df_fe["ServiceCount"]+
    df_fe["StreamingCount"]+
    df_fe["IsAutoPay"]
)

In [12]:
df_fe["Contract_Tenure_Risk"] = (
    (df_fe["Contract"] == "Month-to-month").astype(int) *
    (df_fe["Tenure Months"] < 12).astype(int)
)

In [13]:
df_fe

,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,...,TenureGroup,SpendCategory,AvgSpendPerMonth,HighRiskCustomer,ServiceCount,StreamingCount,IsElectronicCheck,IsAutoPay,TotalEngagement,Contract_Tenure_Risk
0,33.964131,-118.272783,Male,No,No,No,2,Yes,No,DSL,...,New,Medium,1.971741,False,2,0,0,0,2,1
1,34.059281,-118.307420,Female,No,No,Yes,2,Yes,No,Fiber optic,...,New,High,2.115063,False,0,0,1,0,0,1
2,34.048013,-118.293953,Female,No,No,Yes,8,Yes,Yes,Fiber optic,...,New,High,8.152012,False,1,2,1,0,3,1
3,34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,...,Long,VeryHigh,28.790643,False,2,2,1,0,4,0
4,34.039224,-118.266293,Male,No,No,Yes,49,Yes,Yes,Fiber optic,...,Very Long,VeryHigh,48.102197,False,2,2,0,1,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7027,34.341737,-116.539416,Female,No,No,No,72,Yes,No,No,...,Very Long,Low,64.081264,False,0,0,0,1,1,0
7028,34.667815,-117.536183,Male,No,Yes,Yes,24,Yes,Yes,DSL,...,Mid,High,23.199301,False,3,2,0,0,5,0
7029,34.559882,-115.637164,Female,No,Yes,Yes,72,Yes,Yes,Fiber optic,...,Very Long,VeryHigh,70.661228,False,2,2,0,1,5,0
7030,34.167800,-116.864330,Female,No,Yes,Yes,11,No,No phone service,DSL,...,New,Low,11.321895,False,1,0,1,0,1,1


In [15]:
df_fe.to_csv(r"C:\Users\HEMANATH\Desktop\customer-churn-prediction\data\processed\featured_data.csv")

In [16]:
cat_cols=df_fe.select_dtypes(include="object").columns
cat_cols

Index(['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service',
       'Multiple Lines', 'Internet Service', 'Online Security',
       'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV',
       'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method'],
      dtype='object')

In [17]:
nums_cols=df_fe.select_dtypes(exclude="object").columns
nums_cols

Index(['Latitude', 'Longitude', 'Tenure Months', 'Monthly Charges',
       'Total Charges', 'Churn Value', 'TenureGroup', 'SpendCategory',
       'AvgSpendPerMonth', 'HighRiskCustomer', 'ServiceCount',
       'StreamingCount', 'IsElectronicCheck', 'IsAutoPay', 'TotalEngagement',
       'Contract_Tenure_Risk'],
      dtype='object')